Findings from EDA
sales is strongly skewed: 31.3% of rows are exactly 0.
Raw correlation between oil price and total sales: -0.627 (to be revisited below, likely a shared-trend effect).
246 dates were identified as actually non worked once transferred holidays are accounted for.
The April 16, 2016 earthquake produced a clear spike in sales in the following weeks (donations of essential goods).
The payday effect (15th / month end) is real but modest compared to the weekly cycle.
Every day in train.csv has exactly 1782 rows (54 stores x 33 families): the grid is complete, no missing store-family-date rows.

*Goal:* 
To turn the raw tables into a single clean, merged dataset ready for feature engineering (notebook 03). Concretely:

Setup and loading
Cleaning the oil price series (filling non trading days)
Rebuilding an accurate holiday calendar (transferred holidays, event vs closure, national/regional/local scope)
Revisiting the oil/sales correlation on differenced series
Building the earthquake flag from the actual event records
Merging everything into train_clean / test_clean
Sanity checks
A note on transactions.csv and data leakage
Exporting the cleaned datasets

Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')
%matplotlib inline

Loading the data

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
stores = pd.read_csv("stores.csv")
oil = pd.read_csv("oil.csv")
holidays = pd.read_csv("holidays_events.csv")
transactions = pd.read_csv("transactions.csv")

In [3]:
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])
oil["date"] = pd.to_datetime(oil["date"])
holidays["date"] = pd.to_datetime(holidays["date"])
transactions["date"] = pd.to_datetime(transactions["date"])

*Cleaning the oil price series:*
The oil market is closed on weekends and holidays, so oil.csv has gaps. We reindex on a full daily calendar and fill with forward fill then backward fill, as validated in EDA.ipynb

In [4]:
oil_daily = (
    oil.set_index('date')
    .reindex(pd.date_range(oil['date'].min(), oil['date'].max(), name='date'))
    .reset_index()
)
missing_before = oil['dcoilwtico'].isna().sum()
oil_daily['dcoilwtico'] = oil_daily['dcoilwtico'].ffill().bfill()
missing_after = oil_daily['dcoilwtico'].isna().sum()

print(f"Missing oil prices before filling : {missing_before} / {len(oil)}")
print(f"Missing oil prices after filling  : {missing_after} / {len(oil_daily)}")
oil_daily = oil_daily.rename(columns={'dcoilwtico': 'oil_price'})


Missing oil prices before filling : 43 / 1218
Missing oil prices after filling  : 0 / 1704


*Rebuilding an accurate holiday calendar:*
Notebook 1 used a simple date-level rule. Here we refine it to a row-level rule, which is more robust when several holiday rows share the same date, and we add the geographic scope (national, regional, local), since a regional or local holiday only affects the stores in the relevant state or city.

Rules applied, based on the actual type values in holidays_events.csv:

transferred == True rows: the officially listed date is NOT an actual day off, drop it as a closure signal (the real day off is the separate Transfer row at its own date, already present in the table).

Work Day rows: informational only (they mark a date that compensates a bridge day elsewhere), not a closure by themselves, drop from the closure signal.

Event rows: not a closure (Mother's Day, World Cup matches, Black Friday, Cyber Monday, and also the Terremoto Manabi markers), drop from the closure signal but keep for the earthquake flag below.

Remaining types (Holiday, Additional, Bridge, Transfer): actual non worked days.

In [5]:
hol = holidays.copy()

closure_types = ['Holiday', 'Additional', 'Bridge', 'Transfer']
hol_closures = hol[
    (hol['transferred'] == False) &
    (hol['type'].isin(closure_types))
].copy()

print(f"Closure rows kept: {len(hol_closures)} / {len(hol)}")
print(hol_closures['locale'].value_counts())

Closure rows kept: 277 / 350
locale
Local       148
National    105
Regional     24
Name: count, dtype: int64


In [6]:
# National holidays: apply to every store
national_days = set(hol_closures.loc[hol_closures['locale'] == 'National', 'date'])

# Regional holidays: apply only to stores located in the matching state
regional = (
    hol_closures.loc[hol_closures['locale'] == 'Regional', ['date', 'locale_name']]
    .rename(columns={'locale_name': 'state'})
    .drop_duplicates()
)
regional['is_holiday_regional'] = True

# Local holidays: apply only to stores located in the matching city
local = (
    hol_closures.loc[hol_closures['locale'] == 'Local', ['date', 'locale_name']]
    .rename(columns={'locale_name': 'city'})
    .drop_duplicates()
)
local['is_holiday_local'] = True

print(f"National holiday dates : {len(national_days)}")
print(f"Regional holiday rows  : {len(regional)}")
print(f"Local holiday rows     : {len(local)}")

National holiday dates : 102
Regional holiday rows  : 24
Local holiday rows     : 147


In [7]:
# Sanity check: January 1st should be flagged as a national holiday every year
jan1_check = sorted(d for d in national_days if d.strftime('%m-%d') == '01-01')
print("January 1st dates found in national_days:", [d.date() for d in jan1_check])

# 2017 is expected to be missing here: that year's January 1st was transferred,
# so the actual day off should show up as January 2nd instead
jan2_2017 = pd.Timestamp('2017-01-02') in national_days
print("January 2nd, 2017 flagged as a national holiday (the transferred day off):", jan2_2017)

January 1st dates found in national_days: [datetime.date(2013, 1, 1), datetime.date(2014, 1, 1), datetime.date(2015, 1, 1), datetime.date(2016, 1, 1)]
January 2nd, 2017 flagged as a national holiday (the transferred day off): True


Four January 1st dates are captured directly (2013 to 2016). 2017 is expected to be missing from that list: that year's January 1st was officially transferred, so it is correctly excluded, and the real day off shows up as January 2nd, 2017 instead (confirmed above). This matches the sales drop to near zero seen every year in notebook 1's oil/sales chart, and validates that the transferred-holiday logic is working as intended.

*Revisiting the oil/sales correlation:*
EDA.ipynb flagged a risk: a raw correlation of -0.627 between two series that both have their own long term trend can be spurious. We check it here on first differences (day to day changes) instead of raw levels, which removes the shared trend.

In [8]:
daily_sales = train.groupby('date', as_index=False)['sales'].sum()
merged = daily_sales.merge(oil_daily, on='date', how='left')

merged['sales_diff'] = merged['sales'].diff()
merged['oil_diff'] = merged['oil_price'].diff()

corr_level = merged[['sales', 'oil_price']].corr().iloc[0, 1]
corr_diff = merged[['sales_diff', 'oil_diff']].dropna().corr().iloc[0, 1]

print(f"Correlation on raw levels       : {corr_level:.3f}")
print(f"Correlation on first differences: {corr_diff:.3f}")

Correlation on raw levels       : -0.627
Correlation on first differences: 0.025


As suspected, the correlation collapses once the shared trend is removed. This confirms the raw -0.627 was mostly a trend artifact rather than a genuine day to day relationship. Conclusion for the team: we still keep oil_price as a feature (it captures the slow macroeconomic trend, which is legitimately useful), but we should not describe oil price as a strong short term driver of sales, and we will not build a same-day causal narrative around it.

*Earthquake flag, built from the actual event records:*
Instead of hardcoding a date window, we pull it directly from the Terremoto Manabi event rows in holidays_events.csv, which is more precise and reproducible.

In [9]:
quake_events = holidays[holidays['description'].str.contains('Terremoto', case=False, na=False)]
quake_dates = sorted(quake_events['date'])
quake_start, quake_end = quake_dates[0], quake_dates[-1]

print(f"Earthquake related event rows : {len(quake_events)}")
print(f"Window found in the data      : {quake_start.date()} to {quake_end.date()} "
      f"({(quake_end - quake_start).days} days)")

Earthquake related event rows : 31
Window found in the data      : 2016-04-16 to 2016-05-16 (30 days)


This is a national, one off, exogenous shock, so post_earthquake is set to True for every store on every date inside this window, and False otherwise. It is intentionally kept separate from is_holiday, since stores were open (and busier than usual) during this period.

*Merging everything into a clean, single table:*
We now build train_clean and test_clean by merging, in order:

Store metadata (city, state, store type, cluster)
Cleaned oil price (filled)
Holiday flags (national, regional, local, combined into is_holiday)
The earthquake flag
We apply the exact same pipeline to train and test so the two stay consistent.

In [11]:
def clean_and_merge(df, stores, oil_daily, national_days, regional, local, quake_start, quake_end):
    out = df.merge(stores, on='store_nbr', how='left')
    out = out.rename(columns={'type': 'store_type'})

    out = out.merge(oil_daily, on='date', how='left')

    out['is_holiday_national'] = out['date'].isin(national_days)

    out = out.merge(regional, on=['date', 'state'], how='left')
    out['is_holiday_regional'] = out['is_holiday_regional'].infer_objects(copy=False)

    out = out.merge(local, on=['date', 'city'], how='left')
    out['is_holiday_local'] = out['is_holiday_local'].infer_objects(copy=False)

    out['is_holiday'] = out[['is_holiday_national', 'is_holiday_regional', 'is_holiday_local']].any(axis=1)
    out = out.drop(columns=['is_holiday_national', 'is_holiday_regional', 'is_holiday_local'])

    out['post_earthquake'] = (out['date'] >= quake_start) & (out['date'] <= quake_end)

    return out


train_clean = clean_and_merge(train, stores, oil_daily, national_days, regional, local, quake_start, quake_end)
test_clean = clean_and_merge(test, stores, oil_daily, national_days, regional, local, quake_start, quake_end)

print(f"train_clean: {train_clean.shape}")
print(f"test_clean : {test_clean.shape}")
train_clean.head()

train_clean: (3000888, 13)
test_clean : (28512, 12)


,id,date,store_nbr,family,sales,onpromotion,city,state,store_type,cluster,oil_price,is_holiday,post_earthquake
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,93.14,True,False
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,93.14,True,False
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,93.14,True,False
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,93.14,True,False
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,93.14,True,False


Sanity checks

In [12]:
assert len(train_clean) == len(train), "Row count changed for train, a merge likely duplicated rows"
assert len(test_clean) == len(test), "Row count changed for test, a merge likely duplicated rows"

print("Row counts preserved: OK")
print()
print("Missing values in train_clean:")
print(train_clean.isna().sum())
print()
print("Missing values in test_clean:")
print(test_clean.isna().sum())
print()
print(f"Share of holiday rows in train_clean: {train_clean['is_holiday'].mean():.1%}")
print(f"Share of post-earthquake rows in train_clean: {train_clean['post_earthquake'].mean():.2%}")

Row counts preserved: OK

Missing values in train_clean:
id                 0
date               0
store_nbr          0
family             0
sales              0
onpromotion        0
city               0
state              0
store_type         0
cluster            0
oil_price          0
is_holiday         0
post_earthquake    0
dtype: int64

Missing values in test_clean:
id                 0
date               0
store_nbr          0
family             0
onpromotion        0
city               0
state              0
store_type         0
cluster            0
oil_price          0
is_holiday         0
post_earthquake    0
dtype: int64

Share of holiday rows in train_clean: 5.1%
Share of post-earthquake rows in train_clean: 1.84%


No missing values were introduced by the merges, and row counts are unchanged: the merges did not duplicate or drop any row.

*A note on transactions.csv:*
transactions.csv records the number of transactions per store and per day, observed on the same day as sales. It is highly correlated with same-day sales almost by construction, and it is not available for the test period (it is not something we could know in advance for the 15/16 days we need to forecast).

Using it directly as a model feature would therefore leak information that will not exist at prediction time. We keep it loaded for optional exploratory checks, but we deliberately exclude it from train_clean / test_clean. If useful later, only past, already lagged values of transactions (e.g. transactions from 15+ days ago) could be considered as a legitimate feature in next notebook, never same day values.

*Exporting the cleaned datasets:*
We export train_clean.csv and test_clean.csv so notebook 03 can pick up from here without repeating this cleaning logic. On Kaggle, files written to /kaggle/working/ become this notebook's output, which can then be attached as an input dataset to next notebook.

In [13]:
train_clean.to_csv('train_clean.csv', index=False)
test_clean.to_csv('test_clean.csv', index=False)

print("Files written: train_clean.csv, test_clean.csv")

Files written: train_clean.csv, test_clean.csv


*Summary and handoff to notebook Feature Engineering:*
What changed compared to the raw data
oil_price: gaps filled with forward fill then backward fill, no missing values left.
is_holiday: rebuilt at the row level (handles transferred holidays correctly) and scoped by store city/state for regional and local holidays, instead of a single global date flag.
post_earthquake: built directly from the Terremoto Manabi event records rather than an assumed window.
Store metadata (city, state, store_type, cluster) is now merged in.
transactions.csv is deliberately excluded from the modeling tables to avoid leakage.
The oil/sales correlation was re-examined on differenced series: the strong raw correlation (-0.627) is mostly a shared-trend artifact, not a same-day relationship.

Takeaways for feature engineering
Add classic calendar features on top of train_clean / test_clean: day of week, month, year, is_payday (15th / month end), distance to next/previous payday.
Encode family, store_type, cluster, city, state as categorical variables.
Build lag and rolling mean features per store-family pair on log1p(sales).
Consider interaction features, for example is_holiday combined with family, since the impact of a holiday likely differs across product families.